In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import time

from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Labeled event stream: [EventId, Label] per line, produced by the CLAD-Parser
# (see realtime/Thunderbird to regenerate from the raw Thunderbird log)
df = pd.read_csv("labeled_logs_streamed_tbird_v1.csv")

In [12]:
# Map EventIds to integers
event_vocab = {eid: idx for idx, eid in enumerate(sorted(df["EventId"].unique()))}
df["EventIdx"] = df["EventId"].map(event_vocab)

In [13]:
block_size = 500
sequence_length = 80


In [14]:
block_starts = list(range(0, len(df), block_size))
block_labels = [
    1 if df.iloc[i:i+block_size]["Label"].sum() > 0 else 0
    for i in block_starts
]


In [15]:

train_block_starts, test_block_starts = train_test_split(
    block_starts, test_size=0.2, stratify=block_labels, random_state=42
)


In [16]:
class EventSequenceModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dropout=.3),
            num_layers=2
        )
        self.attn = nn.Linear(embed_dim, 1)
        
        #self.fc = nn.Linear(embed_dim, 2)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(.3),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        batch_size, seq_len = x.size()
        
        x = self.embed(x)  # [batch, seq_len, embed_dim]

        # Add positional encoding (learned embedding)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        pos_emb = self.pos_encoder(positions)  # [batch, seq_len, embed_dim]
        x = x + pos_emb
        
        x = x.permute(1, 0, 2)  # [seq_len, batch, embed_dim]
        x = self.transformer(x)
        #x = x[-1]  # last token
        #x = x.mean(dim=0)
        attn_weights = torch.softmax(self.attn(x), dim=0)  # [seq_len, batch, 1]
        x = (x * attn_weights).sum(dim=0)  # weighted sum
        
        return self.fc(x)


In [29]:
def inject_pattern_noise(sequences, pattern=[999, 999, 999], insert_ratio=0.05):
    noisy_sequences = []

    for seq in sequences:
        noisy_seq = seq.copy()
        if random.random() < insert_ratio:
            insert_pos = random.randint(0, len(seq) - len(pattern))
            noisy_seq[insert_pos:insert_pos + len(pattern)] = pattern
        noisy_sequences.append(noisy_seq)

    return noisy_sequences

In [30]:
import random

def inject_random_noise(
    sequences,
    event_vocab,
    noise_ratio=0.05,
    use_new_pool=False,
    new_pool_size=10,
    new_prefix="E"
):
    noisy_sequences = []

    # Prepare noise source
    vocab_ids = list(event_vocab.values())
    max_id = max(vocab_ids)
    new_ids = list(range(max_id + 1, max_id + 1 + new_pool_size))

    #print(vocab_ids)
    #print(new_ids)
    #print(max(1, int(80 * noise_ratio)))  # Ensure at least one corruption
    for seq in sequences:
        noisy_seq = seq.copy()
        num_noisy = max(1, int(len(seq) * noise_ratio))  # Ensure at least one corruption
        noisy_indices = random.sample(range(len(seq)), num_noisy)

        for idx in noisy_indices:
            if use_new_pool:
                noisy_seq[idx] = random.choice(new_ids)
            else:
                noisy_seq[idx] = random.choice(vocab_ids)

        noisy_sequences.append(noisy_seq)

    return noisy_sequences

In [31]:
def inject_shuffle_noise(sequences, window_size=5, shuffle_ratio=0.05):
    noisy_sequences = []

    for seq in sequences:
        noisy_seq = seq.copy()
        if random.random() < shuffle_ratio:
            start = random.randint(0, len(seq) - window_size)
            window = noisy_seq[start:start + window_size]
            random.shuffle(window)
            noisy_seq[start:start + window_size] = window
        noisy_sequences.append(noisy_seq)

    return noisy_sequences

In [32]:
class TBirdSequenceDatasetNoise(torch.utils.data.Dataset):
    def __init__(self, df, block_starts, sequence_length=80, label_column="Label",
                 event_vocab=None, block_size=500, noise_fn=None):
        self.df = df
        self.block_starts = block_starts
        self.sequence_length = sequence_length
        self.label_column = label_column
        self.event_vocab = event_vocab
        self.block_size = block_size
        self.noise_fn = noise_fn  # Optional noise injection function

        self.sequence_refs = []  # (block_start, local_start)

        for start in block_starts:
            block = df.iloc[start:start + block_size].reset_index(drop=True)
            for i in range(0, len(block) - sequence_length):
                self.sequence_refs.append((start, i))

    def __len__(self):
        return len(self.sequence_refs)

    def __getitem__(self, idx):
        block_start, local_start = self.sequence_refs[idx]
        block = self.df.iloc[block_start:block_start + self.block_size].reset_index(drop=True)
        block["EventIdx"] = block["EventId"].map(self.event_vocab)

        seq = block["EventIdx"].iloc[local_start:local_start + self.sequence_length].tolist()
        labels = block[self.label_column].iloc[local_start:local_start + self.sequence_length].tolist()
        label = 1 if 1 in labels else 0

        # Inject noise if function is provided
        if self.noise_fn:
            seq = self.noise_fn([seq])[0]

        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [33]:
test_dataset = TBirdSequenceDatasetNoise(
    df,
    test_block_starts,
    sequence_length=80,
    label_column="Label",
    event_vocab=event_vocab,
    noise_fn=lambda seqs: inject_random_noise(seqs, event_vocab, noise_ratio=0.03)
)

In [34]:
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:
model = EventSequenceModel(vocab_size=len(event_vocab), embed_dim=64, hidden_dim=128)
model.load_state_dict(torch.load("models/TableIV_TableVII_Fig8_tbird_classifier.pth"))
model.to(device)


<python-env>\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


EventSequenceModel(
  (embed): Embedding(150, 64)
  (pos_encoder): Embedding(150, 64)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (attn): Linear(in_features=64, out_features=1, bias=True)
  (fc): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Line

In [36]:
model.eval()
all_preds = []
all_labels = []

total_time = 0
total_samples = 0
num_batches = 0


In [37]:
with torch.no_grad():
    loop = tqdm(test_loader, desc=f"Epoch {1}", leave=True)
    for batch_x, batch_y in loop:
    #for batch_x, batch_y in test_loader:
        #print(type(batch_x))  # Should be torch.Tensor
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        start = time.time()
        output = model(batch_x)
        #print("Output shape:", output.shape)
        preds = output.argmax(dim=1)
        end = time.time()

        batch_time = end - start
        batch_size = batch_x.size(0)

        total_time += batch_time
        total_samples += batch_size
        num_batches += 1

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch_y.cpu().tolist())


Epoch 1: 100%|███████████████████████████████████████████████████████████████████| 48051/48051 [49:53<00:00, 16.05it/s]


In [38]:
avg_time_per_batch = total_time / num_batches
avg_time_per_sample = total_time / total_samples

print(f"Average inference time per batch: {avg_time_per_batch:.6f} seconds")
print(f"Average inference time per sample: {avg_time_per_sample:.6f} seconds")


Average inference time per batch: 0.002074 seconds
Average inference time per sample: 0.000032 seconds


In [39]:
#print(classification_report(all_labels, all_preds))


In [40]:
from collections import Counter
print("Label counts:", Counter(all_labels))


Label counts: Counter({0: 2081005, 1: 994235})


In [41]:
from collections import Counter
print("Label counts:", Counter(all_preds))

Label counts: Counter({0: 1974073, 1: 1101167})


In [42]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(all_labels, all_preds))


[[1966943  114062]
 [   7130  987105]]


In [43]:
cm = confusion_matrix(all_labels, all_preds)
TN, FP, FN, TP = cm.ravel()


In [44]:
#TP , TN, FP, FN = 79267, 688356,30118, 259

In [45]:
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1_score  = 2 * (precision * recall) / (precision + recall)

print(f"Accuracy:  {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall:    {recall:.5f}")
print(f"F1 Score:  {f1_score:.5f}")


Accuracy:  0.96059
Precision: 0.89642
Recall:    0.99283
F1 Score:  0.94216


In [91]:
TP, TN, FP, FN

(np.int64(987131), np.int64(1966880), np.int64(114125), np.int64(7104))